In [4]:
import numpy as np
import pandas as pd
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM, Bidirectional,
    Dense, Dropout, RepeatVector, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Data
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

X = df["text"].astype(str).values
y = df["label"].astype(int).values

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Text vectorization
max_words = 10000
seq_len = 50

vectorizer = TextVectorization(
    max_tokens=max_words,
    output_sequence_length=seq_len
)

vectorizer.adapt(X_train)

all_predictions = {}

def train_and_evaluate(model, name):
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        # callbacks=[early_stop],
        verbose=1
    )

    y_pred = np.argmax(model.predict(X_test), axis=1)
    acc = accuracy_score(y_test, y_pred)

    all_predictions[name] = y_pred

    print(f"\n{name} accuracy: {acc:.4f}")
    return acc

# LSTM
lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Bi-LSTM
bilstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Encoder-Decoder LSTM
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),

    # Encoder
    LSTM(64),

    # Decoder
    RepeatVector(seq_len),
    LSTM(64, return_sequences=True),

    # Classification
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Training
results = {}

results["LSTM"] = train_and_evaluate(lstm_model, "LSTM")
results["Bi-LSTM"] = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["ED-LSTM"] = train_and_evaluate(ed_lstm_model, "ED-LSTM")

# Overall results
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])

print("\n=== Overall Results ===")
print(results_df)

# Detailed results
label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)

    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly predicted": cm.diagonal(),
        "Total in test set": cm.sum(axis=1),
        "Accuracy per class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed class summary:")
    print(summary)

Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 31s 54ms/step - accuracy: 0.4481 - loss: 1.0195 - val_accuracy: 0.4507 - val_loss: 1.0173
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 28s 55ms/step - accuracy: 0.4512 - loss: 0.9857 - val_accuracy: 0.4460 - val_loss: 0.9619
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 28s 55ms/step - accuracy: 0.4635 - loss: 0.9590 - val_accuracy: 0.4732 - val_loss: 0.9753
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 28s 55ms/step - accuracy: 0.4585 - loss: 0.9795 - val_accuracy: 0.4792 - val_loss: 0.9956
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 40s 53ms/step - accuracy: 0.4820 - loss: 0.9951 - val_accuracy: 0.4805 - val_loss: 0.9963
Epoch 6/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 43s 57ms/step - accuracy: 0.4850 - loss: 0.9930 - val_accuracy: 0.4863 - val_loss: 0.9933
Epoch 7/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 30s 58ms/step - accuracy: 0.4864 - loss: 0.9936 - val_accuracy: 0.4858 - val_loss: 0.9909
Epoch 8/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 39s 54ms/step - accuracy: 0.4900 - loss: 0.9914 - 